# 第 12 章 操作変数法: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/12-instrumental-variables.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 200              # 標本の大きさ
strength = 1.0       # 操作変数の強さ（x = strength·z + v）
endogeneity = 0.5    # 誤差 u と v の相関（内生性）
repetitions = 500    # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 1 つの標本の、OLS と IV

In [ ]:
import numpy as np
import pandas as pd

from lib.chapter03.multiple import fit_ols
from lib.chapter12.iv import (
    draw_iv_sample,
    first_stage_f,
    iv_estimate,
    two_stage_least_squares,
    weak_instrument_experiment,
)

sample = draw_iv_sample(n, strength, endogeneity, np.random.default_rng(seed))
y, x, z = sample["y"], sample["x"], sample["z"]
print(f"OLS の傾き {fit_ols(y, x).coefficients.iloc[1]:.3f}、"
      f"IV の傾き {iv_estimate(y, x, z).slope:.3f}（真の値 1）")
tsls = two_stage_least_squares(y, sample[["x"]], sample[["z"]])
print(f"2SLS の傾き {tsls.coefficients['x']:.3f}、"
      f"標準誤差 {tsls.standard_errors['x']:.3f}")
print(f"第 1 段階の F 値 {first_stage_f(x, sample[['z']]).statistic:.1f}"
      "（10 未満は弱操作変数の目安）")

## 強い操作変数と、指定した強さの比較（内生性は 0.5）

In [ ]:
strong = weak_instrument_experiment(n, 1.0, repetitions, np.random.default_rng(seed))
chosen = weak_instrument_experiment(
    n, strength, repetitions, np.random.default_rng(seed)
)
print(f"強さ 1.0: IV の平均 {strong['iv'].mean():.3f}、"
      f"中央値 {strong['iv'].median():.3f}、F の中央値 {strong['f'].median():.1f}")
print(f"強さ {strength}: IV の平均 {chosen['iv'].mean():.3f}、"
      f"中央値 {chosen['iv'].median():.3f}、F の中央値 {chosen['f'].median():.1f}")

## 図: IV の推定値の分布

In [ ]:
from lib.chapter10.plot import plot_estimate_comparison

plot_estimate_comparison(
    pd.DataFrame(
        {
            "strength 1.0": strong["iv"].clip(-1, 3),
            f"strength {strength}": chosen["iv"].clip(-1, 3),
        }
    ),
    1.0,
)

## 試してみよう

- `strength` を 0.1 にすると、第 1 段階の F 値は 0.2 まで下がり、1 つの標本の IV
  の傾きは 1.648（標準誤差 2.086）と、真の値（1）から大きく外れます。標本を取り直した
  IV の推定値は、平均が 1.378、F の中央値が 2.1 です
- `endogeneity` を 0.0 にすると、OLS の傾きは 1.063 で、IV の 1.068 とほぼ同じです。
  内生性がなければ、OLS で足ります
- `n` を 1000 にすると、IV の傾きの標準誤差は 0.081 から 0.031 に、F 値は 1029.4
  になります。OLS は 1.242 で、標本が大きくなっても、1 に近づきません